# Pulsar timing array and the Hellings–Downs curve

*astro · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/pulsar_timing_hellings_downs)

A sea of nanohertz gravitational waves gently stretches the paths between Earth and many millisecond pulsars. An isotropic background, built from a thousand random plane waves, shifts each pulsar's ticks through the antenna pattern $F^A = \tfrac12 \hat p^i \hat p^j e^A_{ij}/(1 + \hat\Omega\cdot\hat p)$, one einsum over polarisation tensors. Correlating every pair of timing residuals and binning by angle recovers the Hellings–Downs curve, the quadrupolar fingerprint that pulsar timing arrays reported evidence for in 2023.

**Vector form**

$$z_a(t) = \frac{\hat p_a^{\,i}\hat p_a^{\,j}}{2(1 + \hat\Omega\cdot\hat p_a)}\,\Delta h_{ij},\qquad \langle r_a r_b\rangle \propto \Gamma(\zeta_{ab}) = \frac12 - \frac{x}{4} + \frac32\,x\ln x,\quad x = \frac{1 - \cos\zeta_{ab}}{2}$$

**Index (tensor) form**

$$e^{+}_{mij} = m_im_j - n_in_j,\quad e^{\times}_{mij} = m_in_j + n_im_j,\qquad F^A_{ma} = \frac{\tfrac12\,e^A_{mij}\,\hat p_{ai}\hat p_{aj}}{1 + \hat\Omega_{mk}\hat p_{ak}},\qquad r_{at} = F^A_{ma}\,\frac{h_{mA}}{\omega_m}\big[\sin\theta_{mAt} - \sin(\theta_{mAt} - \Delta_{ma})\big],\qquad \rho_{ab} = \frac{r_{at}r_{bt}}{\langle r_{ct}r_{ct}\rangle}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\hat p_{ai}$ | `random_directions(N, gen)` | (N, 3) | unit vector to pulsar $a$, uniform on the sky |
| $\hat\Omega_{mi},\ \psi_m$ | `plane_waves(M, J, span, gen)` | (M, 3), (M,) | propagation direction and polarisation angle of wave $m$ |
| $e^A_{mij}$ | `polarisation_tensors(omega, psi)` | (M, 2, 3, 3) | $+$ and $\times$ tensors built from the transverse frame $m, n \perp \hat\Omega$ |
| $e^A_{mij}\hat p_{ai}\hat p_{aj}$ | `einsum('mAij,ni,nj->mAn', e, p, p)` | (M, 2, N) | $i$ and $j$ summed: the stretch of the Earth–pulsar line |
| $F^A_{ma}$ | `antenna_patterns(p, omega, e)` | (M, 2, N) | antenna pattern; sky average of $\sum_A (F^A)^2$ is exactly $1/3$ |
| $\Delta_{ma}$ | `omega_t[:, None] * L * (1 + omega @ p.T)` | (M, N) | phase lag of the pulsar term, $\omega L_a(1 + \hat\Omega\cdot\hat p_a)/c$ |
| $r_{at}$ | `einsum('mAn,mAt->nt', ...)` | (N, T) | $m$ and $A$ summed: timing residual of pulsar $a$ at epoch $t$ |
| $\rho_{ab}$ | `einsum('...at,...bt->...ab', r, r) / auto` | (K, N, N) | $t$ summed: correlation of every pair, normalised by the mean auto-power |
| $\Gamma(\zeta)$ | `hellings_downs(cos_zeta)` | (B,) | expected correlation, $\Gamma(0^+) = \tfrac12$, minimum $\approx -0.152$ at $\zeta \approx 82.5^\circ$ |

A gravitational wave changes the light-travel time between a pulsar and Earth, so the pulse arrival times drift by tens to hundreds of nanoseconds over years. For a single wave the effect depends on the geometry through $F^{+}$ and $F^{\times}$; for an isotropic background, averaging over directions leaves a correlation that depends only on the angle between the two pulsars. Hellings and Downs (ApJL 265, L39, 1983) computed it: pulsars close on the sky are correlated by $\tfrac12$ (the other half of each pulsar's power is its own uncorrelated pulsar term), pairs about $82.5^\circ$ apart are slightly anti-correlated, and opposite pulsars are again positively correlated by $\tfrac14$. The quadrupolar shape is the signature of a transverse, spin-2 wave: a clock error would give a monopole ($\Gamma = 1$) and an error in the Solar-System ephemeris a dipole ($\Gamma = \cos\zeta$); both are fitted for comparison. The page checks the normalisation two ways: the formula against a Monte Carlo sky integral $\tfrac{3}{8\pi}\oint \sum_A F^A_aF^A_b\,d\Omega$, and the antenna-pattern average $\langle\sum_A (F^A)^2\rangle = 1/3$. One sky scatters noticeably about the curve: with only a few dominant frequencies this "cosmic variance" is real and also affects real arrays; averaging many simulated skies shows the convergence. The fitted amplitude is diluted below 1 by white noise. Research context: in June 2023 the NANOGrav Collaboration (15-year data set, Agazie et al., ApJL 951, L8, 2023) reported evidence for a nanohertz gravitational-wave background with Hellings–Downs correlations (roughly 3σ for the correlations, approximate), alongside the European and Indian PTAs (EPTA + InPTA, A&A 2023), the Parkes PTA (Reardon et al., ApJL 951, L6, 2023) and the Chinese PTA. The likely source is a population of supermassive black-hole binaries, which gives the $h_c \propto f^{-2/3}$ spectrum used here. Simplifications: residuals are not fitted for each pulsar's spin-down (only the mean is removed), white noise only, no pulsar-specific red noise, and a simple normalised cross-correlation instead of the optimal statistic or a full Bayesian analysis.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/pulsar_timing_hellings_downs/main.py
"""Main loop: pulsars + random plane-wave skies → timing residuals → pair correlations → HD vs alternatives."""

from pathlib import Path

import torch

from problems._common.runtime import cli, fast_mode

from . import plot, solve, tensors
from .equation import hd_minimum, hellings_downs, separation_cos


def run(params: dict, out_dir: Path) -> Path:
    # CPU, float64: the pulsar-term phases ω L (1 + Ω̂·p̂)/c are hundreds of radians and need double precision.
    fast = fast_mode()
    gen = torch.Generator().manual_seed(params["seed"])
    n_pulsars = 30 if fast else params["n_pulsars"]
    n_real = 5 if fast else params["n_realisations"]
    n_waves = 200 if fast else params["n_waves"]

    # --- checks of the geometry: antenna-pattern sky average and the HD normalisation ---
    probe = tensors.random_directions(1, gen)[0]
    avg = solve.sky_average_power(probe, 20000 if fast else 200000, gen)
    print(f"sky average of Σ_A (F^A)² for one pulsar: {avg:.4f}  (exact 1/3 = {1 / 3:.4f})")
    zetas = torch.tensor([10.0, 45.0, 82.5, 120.0, 170.0], dtype=torch.float64)
    numeric = solve.hd_from_sky_integral(zetas, 20000 if fast else 200000, gen)
    exact = hellings_downs(torch.cos(torch.deg2rad(zetas)))
    for z, a, b in zip(zetas.tolist(), numeric.tolist(), exact.tolist()):
        print(f"Γ({z:5.1f}°): sky integral (3/8π)∮ΣF_aF_b dΩ = {a:+.4f}   formula = {b:+.4f}")
    z_min, g_min = hd_minimum()
    print(
        f"Γ(0⁺) = {hellings_downs(torch.tensor(1.0)).item():.3f}, Γ(180°) = 0.25, "
        f"minimum Γ = {g_min:.4f} at ζ = {z_min:.1f}°"
    )

    # --- the timing array ---
    pulsars = tensors.random_directions(n_pulsars, gen)  # (N, 3)
    distances = tensors.pulsar_distances(n_pulsars, gen)  # (N,) light-years
    t = tensors.time_grid(params["n_times"], params["span"])  # (T,) years
    print(
        f"\n{n_pulsars} pulsars ({n_pulsars * (n_pulsars - 1) // 2} pairs), {params['span']:.0f} yr, "
        f"{n_real} skies of {n_waves} plane waves, white noise {params['noise']} × GW rms"
    )
    r = solve.simulate(pulsars, distances, t, n_real, n_waves, params["n_freq"], params["noise"], gen)  # (K, N, T)

    cos_zeta = separation_cos(pulsars)
    zeta, rho = solve.pair_correlations(r, cos_zeta)  # (P,), (K, P)
    edges = tensors.bin_edges(params["n_bins"])
    centres = (edges[1:] + edges[:-1]) / 2
    binned = solve.bin_by_angle(zeta, rho, edges)  # (K, B)
    err = solve.bin_errors(binned)

    one_sky = solve.fit_models(centres, binned[0], binned.std(dim=0))
    all_skies = solve.fit_models(centres, binned.mean(0), err)
    for label, fits in (("one sky", one_sky), (f"mean of {n_real} skies", all_skies)):
        text = ",  ".join(f"{name}: A = {a:+.2f}, χ² = {chi:.1f}" for name, (a, chi) in fits.items())
        print(f"{label:16s} ({len(centres)} bins) {text}")
    expected = 1 / (1 + params["noise"] ** 2)
    print(f"HD amplitude {all_skies['Hellings–Downs'][0]:.2f}: noise dilutes it towards 1/(1 + σ²) = {expected:.2f}")
    best = min(all_skies, key=lambda name: all_skies[name][1])
    print(f"best model: {best}")

    a = 0
    sep = torch.rad2deg(torch.acos(cos_zeta[a].clamp(-1, 1)))
    sep[a] = 999.0
    near = int(sep.argmin())
    mid = int((sep - 82.5).abs().argmin())
    return plot.animate(
        pulsars, t, r, zeta, rho, edges, binned, [a, near, mid], out_dir / "pulsar_timing_hellings_downs.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/pulsar_timing_hellings_downs/equation.py
r"""Physics → tensor translation for a pulsar timing array.

A plane gravitational wave travelling along the unit vector Ω̂ has metric perturbation
h_{ij}(t) = Σ_A h_A(t) e^A_{ij}(Ω̂) with polarisation tensors (m, n ⟂ Ω̂, rotated by ψ)

    e^+_{ij} = m_i m_j − n_i n_j,        e^×_{ij} = m_i n_j + n_i m_j.

The pulse frequency of pulsar a (unit vector p̂_a) is shifted by the antenna pattern

    F^A_a = ½ p̂^i_a p̂^j_a e^A_{ij} / (1 + Ω̂·p̂_a)                 i, j summed

    z_a(t) = Σ_A F^A_a [h_A(t) − h_A(t − L_a(1 + Ω̂·p̂_a)/c)]           Earth term − pulsar term

and the timing residual is r_a(t) = ∫ z_a dt. For an isotropic background, the expected correlation of two
pulsars separated by ζ (distinct pulsars, pulsar terms uncorrelated) is the Hellings–Downs curve

    Γ(ζ) = ½ − x/4 + (3/2) x ln x,       x = (1 − cos ζ)/2,

normalised so Γ(0⁺) = ½, Γ(π) = ¼ and Γ_aa = 1 (Earth + pulsar term). Equivalently
Γ(ζ) = (3/8π) ∮ dΩ̂ Σ_A F^A_a F^A_b.
"""

import math

import torch


def polarisation_tensors(omega: torch.Tensor, psi: torch.Tensor) -> torch.Tensor:
    r"""e^A_{ij} for every wave.   omega (…, 3) unit vectors, psi (…,) → (…, 2, 3, 3)."""
    z = torch.tensor([0.0, 0.0, 1.0], dtype=omega.dtype, device=omega.device).expand_as(omega)
    x = torch.tensor([1.0, 0.0, 0.0], dtype=omega.dtype, device=omega.device).expand_as(omega)
    helper = torch.where((omega[..., 2:].abs() > 0.9), x, z)  # any vector not parallel to Ω̂
    u = torch.linalg.cross(omega, helper)
    u = u / u.norm(dim=-1, keepdim=True)
    v = torch.linalg.cross(omega, u)
    c, s = torch.cos(psi)[..., None], torch.sin(psi)[..., None]
    m, n = c * u + s * v, -s * u + c * v  # rotate the transverse frame by the polarisation angle ψ
    plus = torch.einsum("...i,...j->...ij", m, m) - torch.einsum("...i,...j->...ij", n, n)
    cross = torch.einsum("...i,...j->...ij", m, n) + torch.einsum("...i,...j->...ij", n, m)
    return torch.stack([plus, cross], dim=-3)


def antenna_patterns(pulsars: torch.Tensor, omega: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    r"""F^A_{ma} = ½ p̂^i_a p̂^j_a e^A_{mij} / (1 + Ω̂_m·p̂_a) — i and j summed.

    pulsars (N, 3), omega (M, 3), e (M, 2, 3, 3) → (M, 2, N)
    """
    numerator = 0.5 * torch.einsum("mAij,ni,nj->mAn", e, pulsars, pulsars)
    denominator = 1.0 + torch.einsum("mi,ni->mn", omega, pulsars)  # (M, N)
    return numerator / denominator[:, None, :].clamp(min=1e-9)


def separation_cos(pulsars: torch.Tensor) -> torch.Tensor:
    r"""cos ζ_{ab} = p̂^i_a p̂^i_b — i summed.   (N, 3) → (N, N)"""
    return torch.einsum("ai,bi->ab", pulsars, pulsars)


def hellings_downs(cos_zeta: torch.Tensor) -> torch.Tensor:
    r"""Γ(ζ) = ½ − x/4 + (3/2) x ln x with x = (1 − cos ζ)/2   (x ln x → 0 at ζ = 0)."""
    x = (1 - cos_zeta) / 2
    return 0.5 - x / 4 + 1.5 * torch.xlogy(x, x)


def monopole(cos_zeta: torch.Tensor) -> torch.Tensor:
    """Γ = 1 for every pair: a common clock error (a correlated error of the time standard)."""
    return torch.ones_like(cos_zeta)


def dipole(cos_zeta: torch.Tensor) -> torch.Tensor:
    """Γ = cos ζ: an error in the Solar-System ephemeris (Earth's position) looks like this."""
    return cos_zeta


def hd_minimum() -> tuple[float, float]:
    """dΓ/dx = 0 ⇒ ln x = −5/6: returns (ζ_min in degrees, Γ_min = ½ − (3/2) e^{−5/6})."""
    x = math.exp(-5 / 6)
    return math.degrees(math.acos(1 - 2 * x)), 0.5 - 1.5 * x


def residuals(
    f_pattern: torch.Tensor,
    amp: torch.Tensor,
    phase: torch.Tensor,
    omega_t: torch.Tensor,
    delay_phase: torch.Tensor,
    t: torch.Tensor,
) -> torch.Tensor:
    r"""r_{at} = Σ_{m,A} F^A_{ma} (h_A/ω_m) [sin(ω_m t + φ_{mA}) − sin(ω_m t + φ_{mA} − Δ_{ma})]   (∫ of z dt).

    f_pattern (M, 2, N), amp and phase (M, 2), omega_t (M,) angular frequencies, delay_phase Δ (M, N), t (T,)
    → (N, T). Uses sin(θ − Δ) = sin θ cos Δ − cos θ sin Δ so every sum is a matrix product over (m, A).
    """
    theta = omega_t[:, None, None] * t + phase[..., None]  # (M, 2, T)
    weight = f_pattern * (amp / omega_t[:, None])[..., None]  # (M, 2, N)
    earth_minus_pulsar_sin = weight * (1 - torch.cos(delay_phase))[:, None, :]
    pulsar_cos = weight * torch.sin(delay_phase)[:, None, :]
    return torch.einsum("mAn,mAt->nt", earth_minus_pulsar_sin, torch.sin(theta)) + torch.einsum(
        "mAn,mAt->nt", pulsar_cos, torch.cos(theta)
    )


def correlation(r: torch.Tensor) -> torch.Tensor:
    r"""ρ_{ab} = ⟨r̃_a r̃_b⟩_t / ⟨r̃²⟩, r̃ = r − mean, normalised by the array-average auto-power.   (…, N, T) → (…, N, N)

    Every pulsar sees the same expected background power, so one common normaliser keeps ρ an unbiased
    (noise-diluted) estimate of Γ: for pure GW residuals ⟨ρ_{ab}⟩ = Γ(ζ_{ab}) and ρ_{aa} ≈ 1.
    """
    r = r - r.mean(dim=-1, keepdim=True)
    c = torch.einsum("...at,...bt->...ab", r, r) / r.shape[-1]  # t summed
    auto = torch.diagonal(c, dim1=-2, dim2=-1).mean(dim=-1)  # (…,) mean over pulsars of ⟨r̃²⟩
    return c / auto[..., None, None]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/pulsar_timing_hellings_downs/tensors.py
"""Build the concrete tensors: pulsar sky positions and distances, the random plane waves, the time grid.

Units: time in years, frequency in 1/yr, distances in light-years (so L/c is in years).
"""

import math

import torch


def random_directions(n: int, gen: torch.Generator, dtype=torch.float64) -> torch.Tensor:
    """n unit vectors uniform on the sphere (normalised Gaussians): (n, 3)."""
    v = torch.randn(n, 3, generator=gen, dtype=dtype)
    return v / v.norm(dim=-1, keepdim=True)


def pulsar_distances(n: int, gen: torch.Generator) -> torch.Tensor:
    """L_a between about 0.5 and 3 kpc, in light-years (1 kpc ≈ 3262 ly): (n,)."""
    return 3262.0 * (0.5 + 2.5 * torch.rand(n, generator=gen, dtype=torch.float64))


def time_grid(n_times: int, span_years: float) -> torch.Tensor:
    """Observation epochs t_k, evenly spaced over the span: (T,)."""
    return torch.linspace(0.0, span_years, n_times, dtype=torch.float64)


def plane_waves(n_waves: int, n_freq: int, span_years: float, gen: torch.Generator) -> dict:
    """One realisation of an isotropic, Gaussian, red background built from M plane waves.

    Ω̂_m (M, 3): propagation directions, uniform on the sphere
    ψ_m (M,): polarisation angles in [0, π)
    ω_m (M,): angular frequency 2π j/T_span, j = 1 … n_freq (the lowest PTA frequency bins)
    h_{mA} (M, 2), φ_{mA} (M, 2): Gaussian amplitudes with σ ∝ f^{−7/6} (strain spectrum S_h ∝ f^{−7/3},
        the f^{−2/3} characteristic strain of inspiralling supermassive black-hole binaries), random phases
    """
    omega = random_directions(n_waves, gen)
    psi = math.pi * torch.rand(n_waves, generator=gen, dtype=torch.float64)
    j = torch.randint(1, n_freq + 1, (n_waves,), generator=gen).double()
    f = j / span_years
    amp = torch.randn(n_waves, 2, generator=gen, dtype=torch.float64) * f[:, None] ** (-7 / 6)
    phase = 2 * math.pi * torch.rand(n_waves, 2, generator=gen, dtype=torch.float64)
    return {"omega": omega, "psi": psi, "omega_t": 2 * math.pi * f, "amp": amp, "phase": phase}


def bin_edges(n_bins: int) -> torch.Tensor:
    """Angular-separation bin edges in degrees, 0 … 180: (n_bins + 1,)."""
    return torch.linspace(0.0, 180.0, n_bins + 1, dtype=torch.float64)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/pulsar_timing_hellings_downs/solve.py
"""Simulate many sky realisations, cross-correlate every pulsar pair, bin by angle and compare models."""

import torch

from problems._common.progress import steps

from . import tensors
from .equation import (
    antenna_patterns,
    correlation,
    dipole,
    hellings_downs,
    monopole,
    polarisation_tensors,
    residuals,
)


def simulate(
    pulsars: torch.Tensor,
    distances: torch.Tensor,
    t: torch.Tensor,
    n_real: int,
    n_waves: int,
    n_freq: int,
    noise: float,
    gen: torch.Generator,
) -> torch.Tensor:
    """Timing residuals r_{kat} for K independent skies: (K, N, T).

    The GW part is scaled so its rms over all pulsars is 1; white noise of rms `noise` is added on top.
    """
    span = float(t[-1] - t[0])
    out = []
    for _ in steps(n_real, "sky realisations"):
        w = tensors.plane_waves(n_waves, n_freq, span, gen)
        e = polarisation_tensors(w["omega"], w["psi"])  # (M, 2, 3, 3)
        f = antenna_patterns(pulsars, w["omega"], e)  # (M, 2, N)
        tau = distances * (1 + torch.einsum("mi,ni->mn", w["omega"], pulsars))  # (M, N) pulsar-term delay
        r = residuals(f, w["amp"], w["phase"], w["omega_t"], w["omega_t"][:, None] * tau, t)  # (N, T)
        r = r - r.mean(dim=-1, keepdim=True)
        r = r / r.pow(2).mean().sqrt()
        out.append(r + noise * torch.randn(r.shape, generator=gen, dtype=r.dtype))
    return torch.stack(out)


def pair_correlations(r: torch.Tensor, cos_zeta: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Correlations of every distinct pair a < b: (ζ in degrees (P,), ρ (K, P))."""
    rho = correlation(r)  # (K, N, N)
    a, b = torch.triu_indices(r.shape[1], r.shape[1], offset=1)
    return torch.rad2deg(torch.acos(cos_zeta[a, b].clamp(-1, 1))), rho[:, a, b]


def bin_by_angle(zeta: torch.Tensor, rho: torch.Tensor, edges: torch.Tensor) -> torch.Tensor:
    """Mean correlation of the pairs in each angular bin, per realisation: (K, B)."""
    which = torch.bucketize(zeta, edges[1:-1])  # (P,) bin index of each pair
    onehot = torch.nn.functional.one_hot(which, len(edges) - 1).double()  # (P, B)
    return torch.einsum("kp,pb->kb", rho, onehot) / onehot.sum(0).clamp(min=1)


def fit_models(centres_deg: torch.Tensor, y: torch.Tensor, err: torch.Tensor) -> dict:
    """Weighted least-squares amplitude A for ρ = A·Γ(ζ) with Γ = HD, monopole, dipole; returns (A, χ²) each."""
    c = torch.cos(torch.deg2rad(centres_deg))
    w = 1.0 / err**2
    fits = {}
    for name, model in (("Hellings–Downs", hellings_downs), ("monopole", monopole), ("dipole", dipole)):
        g = model(c)
        amp = (w * y * g).sum() / (w * g * g).sum()
        fits[name] = (amp.item(), (w * (y - amp * g) ** 2).sum().item())
    return fits


def bin_errors(binned: torch.Tensor) -> torch.Tensor:
    """Standard error of the mean over realisations, per bin: (B,). Needs K ≥ 2."""
    return binned.std(dim=0) / binned.shape[0] ** 0.5


def sky_average_power(pulsar: torch.Tensor, n_dirs: int, gen: torch.Generator) -> float:
    """⟨Σ_A (F^A)²⟩ over isotropic wave directions for one pulsar (analytically 1/3)."""
    omega = tensors.random_directions(n_dirs, gen)
    psi = torch.pi * torch.rand(n_dirs, generator=gen, dtype=torch.float64)
    f = antenna_patterns(pulsar[None], omega, polarisation_tensors(omega, psi))  # (M, 2, 1)
    return f.pow(2).sum(1).mean().item()


def hd_from_sky_integral(zeta_deg: torch.Tensor, n_dirs: int, gen: torch.Generator) -> torch.Tensor:
    """Γ(ζ) = (3/8π) ∮ dΩ Σ_A F^A_a F^A_b, by Monte Carlo over directions (Earth terms only): (Z,)."""
    z = torch.deg2rad(zeta_deg.double())
    a = torch.tensor([[0.0, 0.0, 1.0]], dtype=torch.float64).expand(len(z), 3)
    b = torch.stack([torch.sin(z), torch.zeros_like(z), torch.cos(z)], -1)  # (Z, 3)
    omega = tensors.random_directions(n_dirs, gen)
    psi = torch.pi * torch.rand(n_dirs, generator=gen, dtype=torch.float64)
    e = polarisation_tensors(omega, psi)
    fa, fb = antenna_patterns(a, omega, e), antenna_patterns(b, omega, e)  # (M, 2, Z)
    return 1.5 * torch.einsum("mAz,mAz->z", fa, fb) / n_dirs  # (3/8π)·4π·mean = (3/2)·mean


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/pulsar_timing_hellings_downs/plot.py
"""Animate sky after sky: the pulsar array, a few residual curves and the binned correlation converging to HD."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from . import solve
from .equation import hellings_downs


def animate(
    pulsars: torch.Tensor,
    t: torch.Tensor,
    r: torch.Tensor,
    zeta: torch.Tensor,
    rho: torch.Tensor,
    edges: torch.Tensor,
    binned: torch.Tensor,
    shown: list[int],
    path: Path,
) -> Path:
    k_total = r.shape[0]
    p = pulsars.numpy()
    lon, lat = np.arctan2(p[:, 1], p[:, 0]), np.arcsin(p[:, 2])
    centres = (edges[1:] + edges[:-1]) / 2
    spread = binned.std(dim=0)  # (B,) scatter of a bin between skies
    zz = np.linspace(0, 180, 361)
    hd = hellings_downs(torch.cos(torch.deg2rad(torch.tensor(zz)))).numpy()
    t_np, r_np, rho_np, zeta_np = t.numpy(), r.numpy(), rho.numpy(), zeta.numpy()
    some = np.random.default_rng(0).permutation(len(zeta_np))[:1500]  # pairs drawn as dots (keeps the GIF small)
    mid = len(t_np) // 2

    fig = plt.figure(figsize=(11.5, 3.9))
    ax_m = fig.add_subplot(1, 3, 1, projection="mollweide")
    ax_r = fig.add_subplot(1, 3, 2)
    ax_c = fig.add_subplot(1, 3, 3)

    lim = 1.5 * float(r_np[:, :, mid].std())
    sky = ax_m.scatter(lon, lat, c=r_np[0, :, mid], cmap="coolwarm", vmin=-lim, vmax=lim, s=24, edgecolors="#374151")
    for a, mark in zip(shown, ("A", "B", "C")):
        ax_m.text(lon[a], lat[a] + 0.12, mark, fontsize=8, ha="center", color="#111827")
    ax_m.set_xticklabels([])
    ax_m.tick_params(labelsize=7)
    ax_m.set_title(f"{len(p)} pulsars, colour = residual at t = {t_np[mid]:.1f} yr", fontsize=9)

    lines = [ax_r.plot([], [], lw=1.5, label=m)[0] for m in ("A", "B", "C")]
    ax_r.set_xlim(t_np[0], t_np[-1])
    lim_r = float(np.abs(r_np[:, shown]).max()) * 1.05
    ax_r.set_ylim(-lim_r, lim_r)
    ax_r.set_xlabel("time (yr)")
    ax_r.set_ylabel("timing residual (GW rms = 1)")
    ang = [np.degrees(np.arccos(np.clip(p[shown[0]] @ p[b], -1, 1))) for b in shown[1:]]
    ax_r.legend(loc="lower left", fontsize=7, title=f"ζ(A,B) = {ang[0]:.0f}°, ζ(A,C) = {ang[1]:.0f}°", title_fontsize=7)
    title_r = ax_r.set_title("sky realisation", fontsize=9)

    pairs = ax_c.scatter(zeta_np[some], rho_np[0, some], s=2, color="#d1d5db", zorder=1, label="pairs, this sky")
    ax_c.plot(zz, hd, color="#dc2626", lw=2, zorder=3, label="Hellings–Downs Γ(ζ)")
    (hd_fit,) = ax_c.plot([], [], color="#dc2626", lw=1, ls=":", zorder=3, label="A·Γ fit")
    (mono,) = ax_c.plot([], [], color="#f59e0b", lw=1.2, ls="--", zorder=2, label="monopole fit")
    (dip,) = ax_c.plot([], [], color="#16a34a", lw=1.2, ls="--", zorder=2, label="dipole fit")
    bars = [None]
    ax_c.axhline(0, color="#9ca3af", lw=1)
    ax_c.set_xlim(0, 180)
    ax_c.set_ylim(-0.7, 1.05)
    ax_c.set_xticks([0, 45, 90, 135, 180])
    ax_c.set_xlabel("angular separation ζ (deg)")
    ax_c.set_ylabel("correlation ρ")
    ax_c.legend(loc="upper center", fontsize=6.5, ncol=2, framealpha=0.95)
    title_c = ax_c.set_title("mean of skies", fontsize=9)

    def update(k: int):
        sky.set_array(r_np[k, :, mid])
        for line, a in zip(lines, shown):
            line.set_data(t_np, r_np[k, a])
        title_r.set_text(f"sky realisation {k + 1} of {k_total}")
        pairs.set_offsets(np.column_stack([zeta_np[some], rho_np[k, some]]))
        mean = binned[: k + 1].mean(0)
        err = spread / (k + 1) ** 0.5
        fits = solve.fit_models(centres, mean, err)
        if bars[0] is not None:
            bars[0].remove()
        bars[0] = ax_c.errorbar(
            centres.numpy(), mean.numpy(), yerr=err.numpy(), fmt="o", ms=4, color="#2563eb", capsize=2, zorder=4
        )
        c = np.cos(np.radians(zz))
        hd_fit.set_data(zz, fits["Hellings–Downs"][0] * hd)
        mono.set_data(zz, fits["monopole"][0] * np.ones_like(zz))
        dip.set_data(zz, fits["dipole"][0] * c)
        chi = {name: v[1] for name, v in fits.items()}
        title_c.set_text(
            f"mean of {k + 1} skies, χ²: HD {chi['Hellings–Downs']:.0f}, "
            f"mono {chi['monopole']:.0f}, dip {chi['dipole']:.0f}"
        )

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(k_total, k_total), path, fps=3)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_pulsars': 100,
    'n_realisations': 30,
    'n_waves': 1000,
    'n_freq': 8,
    'span': 15.0,
    'n_times': 128,
    'noise': 0.3,
    'n_bins': 12,
    'seed': 0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.pulsar_timing_hellings_downs")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.pulsar_timing_hellings_downs.main")
problem_dir = Path("problems/astro/pulsar_timing_hellings_downs")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
